## 12. Document the Exact Windows Command and Current Limitations

Capture the precise command needed to run the analytical engine from the project root on Windows, and document any remaining errors or limitations without weakening validation or changing the verified/unverified safeguards.

Current limitation: the terminal environment is still experiencing an invalid-syntax issue before Python command execution begins, so the project must be run from a path-safe environment or a Python launcher that avoids the misparsed Windows path string.

In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
out_dir = repo_root / 'data' / 'processed'
files = sorted(out_dir.glob('*.csv'))
print('output csv files:', [p.name for p in files])

for path in files:
    df = pd.read_csv(path)
    print(path.name, 'rows=', len(df), 'cols=', list(df.columns[:10]), 'nulls=', int(df.isna().sum().sum()), 'duplicates=', int(df.duplicated().sum()))
    if 'source_dataset' in df.columns:
        print('source_dataset values:', df['source_dataset'].dropna().unique()[:5])
    if 'verification_status' in df.columns:
        print('verification_status values:', df['verification_status'].dropna().unique()[:5])


## 11. Validate Output Files, Row Counts, Schemas, Nulls, Duplicates, and Provenance

Check the generated outputs for file existence, row counts, column names, nulls, duplicate records, impossible values, provenance fields, and other integrity checks required by the project.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
print('repo_root:', repo_root)
print('pytest path-safe command candidate: python -m pytest -q')

# We will not claim pass/fail unless the command actually runs in this environment.
# This call uses a safe project-root relative execution context.
cmd = [sys.executable, '-m', 'pytest', '-q']
print('running command:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(repo_root), capture_output=True, text=True)
print('returncode:', result.returncode)
print('stdout:\n' + result.stdout[:4000])
print('stderr:\n' + result.stderr[:4000])


## 10. Run the Real Test Suite with a Path-Safe Command

Execute the actual test suite from the project root using a path-safe command such as python -m pytest or py -m pytest, and record whether tests run successfully and how many pass or fail.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from analytics.kpi_engine import get_kpi_registry, calculate_kpi_status

registry = get_kpi_registry()
status = calculate_kpi_status(pd.read_csv(repo_root / 'data' / 'processed' / 'processed_verified_product_catalog.csv'))

print('all registry statuses sample:')
print(registry[['kpi_name', 'verified_status']].head(10).to_string(index=False))
print('\nverified counts by status:')
print(status['verified_status'].value_counts(dropna=False).to_string())
print('\nPending registry entries count:', status[status['verified_status'].str.contains('PENDING VERIFIED DATA', na=False)].shape[0])
print('Verified entries count:', status[status['verified_status'] == 'verified'].shape[0])


## 9. Enforce Verified-Only KPI Classification and Pending Safeguards

Confirm that product-level KPIs calculate from validated data, while market, OEM, state, and unit-economics KPI categories remain clearly marked as PENDING VERIFIED DATA instead of silently computing from proxy sources.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from analytics.kpi_engine import get_kpi_registry, calculate_kpi_status
from analytics.pricing_analysis import compute_price_percentile, compute_value_score
from analytics.positioning_analysis import generate_price_vs_range_positioning, identify_competitive_whitespace

product_df = pd.read_csv(repo_root / 'data' / 'processed' / 'processed_verified_product_catalog.csv')
registry = get_kpi_registry()
status = calculate_kpi_status(product_df)

print('kpi_registry rows:', len(registry))
print('kpi_status rows:', len(status))
print('product-level verified KPI rows:', status[status['verified_status'] == 'verified'].shape[0])
print('pending KPI rows:', status[status['verified_status'].str.contains('PENDING VERIFIED DATA', na=False)].shape[0])
print('pricing percentile columns:', list(compute_price_percentile(product_df).columns))
print('positioning rows:', len(generate_price_vs_range_positioning(product_df)))
print('whitespace rows:', len(identify_competitive_whitespace(product_df)))


## 8. Generate Pricing, Positioning, and KPI Registry Outputs

Generate pricing metrics, product positioning data, and KPI registry artifacts using the verified catalog inputs only, while retaining the established naming conventions if the implementation uses slightly different filenames.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from analytics.competitive_analysis import normalize_verified_product_schema, compute_manufacturer_count, compute_product_count, compute_product_category_distribution, compute_manufacturer_portfolio_analysis

product_df = pd.read_csv(repo_root / 'data' / 'processed' / 'processed_verified_product_catalog.csv')
product_df = normalize_verified_product_schema(product_df)

summary = pd.DataFrame([
    {'metric': 'manufacturer_count', 'value': compute_manufacturer_count(product_df), 'status': 'verified'},
    {'metric': 'product_count', 'value': compute_product_count(product_df), 'status': 'verified'},
])
print(summary)
print('category distribution rows:', len(compute_product_category_distribution(product_df)))
print('manufacturer portfolio rows:', len(compute_manufacturer_portfolio_analysis(product_df)))


## 7. Generate Verified Competitive Product and Manufacturer Outputs

Create the product-level and manufacturer-level summary outputs expected by the project, preserving all verified-only data constraints and excluding any unverified or proxy-derived market results.

In [ ]:
from analytics.competitive_analysis import (
    normalize_verified_product_schema,
    compute_manufacturer_count,
    compute_product_count,
    compute_product_category_distribution,
    compute_price_comparison,
    compute_battery_comparison,
    compute_range_comparison,
    compute_top_speed_comparison,
    compute_motor_power_comparison,
    compute_price_per_km,
    compute_price_per_kwh,
    compute_range_realization,
    compute_manufacturer_portfolio_analysis,
    rank_products,
)
from analytics.pricing_analysis import (
    compute_ex_showroom_price,
    compute_effective_price_after_subsidy,
    compute_price_per_certified_km,
    compute_price_percentile,
    compute_range_percentile,
    compute_battery_percentile,
    compute_value_score,
)
from analytics.positioning_analysis import (
    generate_price_vs_range_positioning,
    generate_price_vs_battery_positioning,
    generate_price_vs_performance_positioning,
    identify_competitive_whitespace,
    product_segment_classification,
)
from analytics.kpi_engine import get_kpi_registry, calculate_kpi_status

repo_root = Path.cwd().resolve()
out_dir = repo_root / 'data' / 'processed'
out_dir.mkdir(exist_ok=True)

product_df = normalize_verified_product_schema(pd.read_csv(repo_root / 'data' / 'processed' / 'processed_verified_product_catalog.csv'))

# Product metrics
compute_product_category_distribution(product_df).to_csv(out_dir / 'competitive_product_metrics.csv', index=False)
compute_manufacturer_portfolio_analysis(product_df).to_csv(out_dir / 'competitive_manufacturer_summary.csv', index=False)

# Pricing metrics
compute_ex_showroom_price(product_df).to_csv(out_dir / 'pricing_metrics.csv', index=False)
compute_effective_price_after_subsidy(product_df).to_csv(out_dir / 'pricing_effective_price_after_subsidy.csv', index=False)
compute_price_per_certified_km(product_df).to_csv(out_dir / 'pricing_price_per_certified_km.csv', index=False)
compute_price_percentile(product_df).to_csv(out_dir / 'pricing_percentile.csv', index=False)
compute_value_score(product_df).to_csv(out_dir / 'pricing_value_score.csv', index=False)

# Positioning outputs
generate_price_vs_range_positioning(product_df).to_csv(out_dir / 'product_positioning.csv', index=False)
generate_price_vs_battery_positioning(product_df).to_csv(out_dir / 'product_positioning_battery.csv', index=False)
identify_competitive_whitespace(product_df).to_csv(out_dir / 'whitespace_analysis.csv', index=False)

# KPI outputs
get_kpi_registry().to_csv(out_dir / 'kpi_registry.csv', index=False)
calculate_kpi_status(product_df).to_csv(out_dir / 'kpi_status.csv', index=False)

# Template outputs
unit_template = pd.DataFrame([{
    'selling_price': None,
    'variable_cost': None,
    'fixed_cost': None,
    'units_sold': None,
    'status': 'PENDING VERIFIED DATA',
    'notes': 'Cost data is not present in the current verified catalog.'
}])
unit_template.to_csv(out_dir / 'unit_economics_template.csv', index=False)

geo_template = pd.DataFrame([{
    'state': 'example_state',
    'market_size': None,
    'ev_penetration': None,
    'growth': None,
    'status': 'PENDING VERIFIED DATA',
    'notes': 'State-level verified market data is not currently available.'
}])
geo_template.to_csv(out_dir / 'geographic_scoring_template.csv', index=False)

print('Generated files in data/processed:')
for p in sorted(out_dir.iterdir()):
    print('-', p.name)


In [ ]:
import sys
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Use the verified raw catalog as the only factual product source.
verified_catalog = repo_root / 'data' / 'raw' / 'verified_product_catalog_2025_2026.csv'
processed_verified_catalog = repo_root / 'data' / 'processed' / 'processed_verified_product_catalog.csv'

print('verified_catalog.exists():', verified_catalog.exists())
print('processed_verified_catalog.exists():', processed_verified_catalog.exists())

if processed_verified_catalog.exists():
    product_df = pd.read_csv(processed_verified_catalog)
else:
    product_df = pd.read_csv(verified_catalog)

print('rows:', len(product_df))
print('head columns:', list(product_df.columns[:10]))

# Safe project root based output generation.
out_dir = repo_root / 'data' / 'processed'
out_dir.mkdir(exist_ok=True)


## 6. Execute the Analytical Engine Against Verified Catalog Data Only

Run the existing Prompt 3 analytical engine using only the verified raw catalog and the verified processed catalog, ensuring the pipeline creates the intended output artifacts without fabricating market, OEM, or state data.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
src_dir = repo_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('module path inserted:', src_dir)
print('exists:', src_dir.exists())

# Verify imports that the analytics modules depend on.
try:
    import pandas as pd
    import numpy as np
    print('pandas/numpy import OK')
except Exception as exc:
    print('pandas/numpy import failed:', exc)

for module_name in ['analytics.competitive_analysis', 'analytics.pricing_analysis', 'analytics.positioning_analysis', 'analytics.kpi_engine']:
    try:
        __import__(module_name)
        print(module_name, 'import OK')
    except Exception as exc:
        print(module_name, 'import failed:', exc)


## 5. Repair Import and Execution Configuration Issues

Resolve missing imports, Python module resolution issues, and test/runtime configuration problems that block execution while keeping the validated safeguards in place and avoiding any new analytical feature development.

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
raw_dir = repo_root / 'data' / 'raw'
processed_dir = repo_root / 'data' / 'processed'

# These are portably constructed and do not embed a Windows username.
print('raw_dir =', raw_dir)
print('processed_dir =', processed_dir)
print('verified_raw_catalog =', raw_dir / 'verified_product_catalog_2025_2026.csv')
print('processed_verified_catalog =', processed_dir / 'processed_verified_product_catalog.csv')

# Avoid using any raw string like r'C:\Users\...' in the project code.
assert 'C:\\Users' not in str(raw_dir)
assert 'C:\\Users' not in str(processed_dir)


## 4. Fix Unsafe String Paths and Environment Assumptions

Correct path construction for the raw data, processed data, and output folders using project-root-relative operations, ensuring that backslashes are not misinterpreted and that paths remain portable across Windows environments.

In [ ]:
from pathlib import Path
import re

repo_root = Path.cwd().resolve()
paths = list(repo_root.rglob('*.py')) + list(repo_root.rglob('*.md'))
unsafe = []
for p in paths:
    text = p.read_text(encoding='utf-8', errors='ignore')
    if re.search(r'"?[A-Za-z]:\\Users\\', text) or re.search(r"'?[A-Za-z]:\\Users\\", text):
        unsafe.append(str(p))

print('unsafe user-specific paths found:', unsafe)

# Safe path strategy: always derive from the repository root and __file__.
base = Path(__file__).resolve().parent if '__file__' in globals() else repo_root
print('safe base path:', base)


## 3. Audit Path Handling for Windows Username and Project Location

Search for unsafe Windows path strings and hardcoded user-specific paths, then replace them with robust pathlib-based logic that works across different usernames and project locations without Unicode escape issues.

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
raw_dir = repo_root / 'data' / 'raw'
processed_dir = repo_root / 'data' / 'processed'

verified_catalog = raw_dir / 'verified_product_catalog_2025_2026.csv'
print('verified catalog exists:', verified_catalog.exists())
print('verified catalog columns:', pd.read_csv(verified_catalog).columns.tolist()[:15])

expected_outputs = [
    'competitive_product_metrics.csv',
    'competitive_manufacturer_summary.csv',
    'pricing_metrics.csv',
    'product_positioning.csv',
    'kpi_registry.csv',
    'kpi_status.csv',
    'unit_economics_template.csv',
    'geographic_scoring_template.csv',
]
print('expected outputs list:', expected_outputs)

print('present processed files:', sorted(p.name for p in processed_dir.iterdir()))

# Read the manifest to confirm that unverified market data is explicitly disallowed.
manifest_path = raw_dir / 'DATASET_MANIFEST.csv'
manifest = pd.read_csv(manifest_path)
print('manifest entries:', manifest[['dataset_name', 'provenance_class', 'eligible_as_observed_analytics']].to_dict('records')[:3])


## 2. Identify Verified Inputs, Outputs, and Missing Dependencies

List the expected output artifacts for the Prompt 3 analytics run, verify which inputs are available from the processed and verified raw catalogs, and inspect imports or module references that may be missing or misconfigured.

The verified raw source is limited to the official product catalog; all market, OEM, state, and unit-economics outputs should remain pending unless verified data exists.

In [ ]:
from pathlib import Path
import pandas as pd

# Confirm the project’s entry point and project-relative layout.
repo_root = Path.cwd().resolve()
print('repo_root:', repo_root)
print('src exists:', (repo_root / 'src').exists())
print('raw exists:', (repo_root / 'data' / 'raw').exists())
print('processed exists:', (repo_root / 'data' / 'processed').exists())

# Confirm the main data-processing entry point.
entry = repo_root / 'src' / 'data_processing.py'
print('data_processing.py exists:', entry.exists())
if entry.exists():
    print('entry point file:', entry)

# Discover the analytics modules present.
analytics_files = sorted((repo_root / 'src' / 'analytics').glob('*.py'))
print('analytics modules:', [p.name for p in analytics_files])

# Discover the tests.
test_files = sorted((repo_root / 'src' / 'tests').glob('*.py'))
print('tests:', [p.name for p in test_files])

# Discover the raw datasets.
raw_files = sorted((repo_root / 'data' / 'raw').glob('*.csv'))
print('raw datasets:', [p.name for p in raw_files])

# Discover processed outputs already present.
processed_files = sorted((repo_root / 'data' / 'processed').glob('*'))
print('processed files:', [p.name for p in processed_files])


## 1. Inspect Repository Structure and Entry Point

Review the project structure under src/, src/analytics/, tests/, data/raw/, and data/processed/ to locate the analytical engine entry point, identify key modules, and confirm how the current implementation is invoked.

This section confirms the engine entry point is the project’s data-processing pipeline and the verified-only analytics functions in src/analytics/.

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd

root = Path.cwd().resolve()
print(f"Project root: {root}")
print(f"Exists: {root.exists()}")
print(f"Files in root: {sorted(p.name for p in root.iterdir())[:15]}")

src_dir = root / 'src'
analytics_dir = src_dir / 'analytics'
raw_dir = root / 'data' / 'raw'
processed_dir = root / 'data' / 'processed'

for path in [src_dir, analytics_dir, raw_dir, processed_dir]:
    print(f"{path}: exists={path.exists()}")

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
print(f"sys.path[0:5] = {sys.path[:5]}")


# PROMPT 3A — Finish and Verify the Analytical Engine

This notebook is the project-scoped verification pass for the existing Prompt 3 analytical engine.
It inspects the repo, fixes Windows path issues, runs the verified-only pipeline, validates KPI status, and records the exact commands and outcomes without inventing any proxy market data.

Objectives:
- Confirm the engine entry point and current data flow.
- Ensure path handling is robust across Windows user names and project locations.
- Run only the verified inventory-based analytics from the raw verified product catalog.
- Keep market/OEM/state/unit-economics outputs explicitly pending where verified data is absent.
- Verify the real test suite executes successfully when the environment is path-safe.
- Record the final evidence for the project handoff.
